# 04d · Chronos-2 Few-Shot (In-Context Learning)

**Owner:** Max · **Reads:** notebook 01 outputs, `chronos2_zs` forecasts from 04a · **Feeds:** 05 (scoring)

**The question:** if Chronos-2 can look at a group of related products while it forecasts, does it do better than forecasting each product alone?

**Few-shot here means in-context learning**, the way it works for ChatGPT: the model is **not retrained**. It gets examples in its input
and uses them on the spot. Chronos-2 calls this *cross-learning*: all the series in one batch are forecast together, and the model shares
information across them. Our "examples" are related products:
- Products are sorted by ID (`FOODS_3_001_CA_1`, `FOODS_3_001_CA_2`, …), so neighbours are **the same item in the other California stores**
  and **nearby items in the same department**.
- Each batch holds 100 series, about 25 items × 4 stores. 100 is the group size the Chronos-2 technical report used.

**Where it should help most:** products with little history (late launches) and slow sellers, where one series alone says little.
The library notes it doesn't always help, which is why we test it.

| Model name in files | What it is |
|---|---|
| `chronos2_fs` | Chronos-2, sales history only, forecasting 100 related products together (cross-learning), no training |

**Where it fits:** zero-shot (04a: nothing from us) → **few-shot (04d: related examples, no training)** → fine-tuned (04c: trained on our sales).

**Windows:** `val_1`, `val_2`, `test` by default. **Output:** `data/forecasts/chronos2_fs_<window>.parquet`, same columns as 04a, so 05 scores it automatically.

**Runtime:** **T4 GPU.** About as long as one zero-shot run in 04a. Run `SAMPLE = True` first, then `SAMPLE = False`. Finished windows are skipped if Colab disconnects.

## 0 · Settings

In [9]:
SAMPLE      = False       # True = 100-product smoke test; False = all ~12,200 California series
WINDOWS     = ["val_1", "val_2", "test"]   # add "bt_4", "bt_3", "bt_2", "bt_1" only if we later want it in 06
OVERWRITE   = False      # True = redo windows that already have a forecast file
GROUP_SIZE  = 100        # series forecast together (one batch = one group of related products)

DRIVE_DIR   = "/content/drive/MyDrive/AML_Term_Project"
DATA_DIR    = f"{DRIVE_DIR}/data"
OUT_DIR     = f"{DATA_DIR}/forecasts_sample" if SAMPLE else f"{DATA_DIR}/forecasts"

MODEL_ID    = "amazon/chronos-2"
MODEL_NAME  = "chronos2_fs"
QUANTILES   = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

## 1 · Install and connect

In [10]:
!pip install -q "chronos-forecasting==2.3.2"

In [11]:
import os, json, time, gc
import numpy as np
import pandas as pd
import torch

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab: using the paths in Settings as-is.")

os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| torch", torch.__version__)
if DEVICE == "cpu":
    print("WARNING: no GPU found. Set Runtime -> Change runtime type -> T4 GPU.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda | torch 2.14.1+cu130


## 2 · Load data and windows (same window list as every notebook)

In [12]:
df = pd.read_parquet(f"{DATA_DIR}/{'ca_sample100' if SAMPLE else 'ca_sales'}.parquet", columns=["unique_id", "ds", "y"])
df["unique_id"] = df["unique_id"].astype(str)
df["ds"] = pd.to_datetime(df["ds"])
# Sorting by ID puts related products next to each other: same item across the 4 stores, then nearby items in the same department.
# Chronos-2 keeps this order when it forms batches, so each batch of GROUP_SIZE is a group of related products.
df = df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

with open(f"{DATA_DIR}/windows.json") as f:
    wjson = json.load(f)
H = wjson["horizon"]

val1_start = pd.Timestamp(wjson["val_1"]["start"])
windows = {}
for k in range(4, 0, -1):
    start = val1_start - pd.Timedelta(days=H * k)
    windows[f"bt_{k}"] = (start, start + pd.Timedelta(days=H - 1))
for name in ["val_1", "val_2", "test"]:
    windows[name] = (pd.Timestamp(wjson[name]["start"]), pd.Timestamp(wjson[name]["end"]))
windows = {w: windows[w] for w in windows if w in WINDOWS}

print(f"{df['unique_id'].nunique():,} series, {len(df):,} rows")
print("First group starts with:", list(df["unique_id"].drop_duplicates()[:8]))
pd.DataFrame({w: {"start": s.date(), "end": e.date()} for w, (s, e) in windows.items()}).T

12,196 series, 18,559,286 rows
First group starts with: ['FOODS_1_001_CA_1', 'FOODS_1_001_CA_2', 'FOODS_1_001_CA_3', 'FOODS_1_001_CA_4', 'FOODS_1_002_CA_1', 'FOODS_1_002_CA_2', 'FOODS_1_002_CA_3', 'FOODS_1_002_CA_4']


,start,end
val_1,2016-02-29,2016-03-27
val_2,2016-03-28,2016-04-24
test,2016-04-25,2016-05-22


## 3 · Load pretrained Chronos-2

In [13]:
from chronos import Chronos2Pipeline
pipeline = Chronos2Pipeline.from_pretrained(MODEL_ID, device_map=DEVICE)
print("Loaded", MODEL_ID)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loaded amazon/chronos-2


## 4 · Few-shot forecast for one window
Same as 04a's zero-shot run, with one change: `cross_learning=True`, so the 100 products in each batch are forecast together and
the model can borrow patterns across them. Only sales **before** the window are used.

In [14]:
def forecast(wname, start, batch_size=GROUP_SIZE):
    context = df[df["ds"] < start]
    while True:
        try:
            out = pipeline.predict_df(
                context, id_column="unique_id", timestamp_column="ds", target="y",
                prediction_length=H, quantile_levels=QUANTILES, batch_size=batch_size,
                cross_learning=True,
            )
            break
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            batch_size //= 2
            print(f"  GPU out of memory -> retrying with groups of {batch_size}")
            if batch_size < 4:
                raise
    res = pd.DataFrame({
        "unique_id": out["unique_id"].astype(str),
        "ds": pd.to_datetime(out["ds"]),
        "model": MODEL_NAME,
        "window": wname,
        "forecast": out["0.5"].clip(lower=0).astype("float32"),
        "mean": out["predictions"].clip(lower=0).astype("float32"),
    })
    for q in QUANTILES:
        res[f"q{q}"] = out[str(q)].clip(lower=0).astype("float32")
    return res, batch_size

## 5 · Run every window (skips windows already saved)

In [15]:
log_rows = []
for wname, (start, end) in windows.items():
    path = f"{OUT_DIR}/{MODEL_NAME}_{wname}.parquet"
    if os.path.exists(path) and not OVERWRITE:
        print(f"skip  {MODEL_NAME} {wname:6s} (already saved)")
        continue
    t0 = time.time()
    res, used = forecast(wname, start)
    res.to_parquet(path, index=False)
    secs = time.time() - t0
    n = res["unique_id"].nunique()
    print(f"done  {MODEL_NAME} {wname:6s} {n:,} series in groups of {used}  {secs/60:5.1f} min")
    log_rows.append({"model": MODEL_NAME, "window": wname, "series": n, "group_size": used,
                     "minutes": round(secs / 60, 1), "device": DEVICE, "sample": SAMPLE,
                     "run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")})
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

if log_rows:
    log_path = f"{OUT_DIR}/run_log.csv"
    new = pd.DataFrame(log_rows)
    new.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
    display(new)

skip  chronos2_fs val_1  (already saved)
skip  chronos2_fs val_2  (already saved)
skip  chronos2_fs test   (already saved)


## 6 · Quick check: few-shot vs. zero-shot
Not the official evaluation (that's 05). Same products, same days. Negative change = few-shot helped.
Split by sales speed and by **new products** (under one year of sales history before the window), where borrowing from related products should matter most.

In [16]:
static = pd.read_parquet(f"{DATA_DIR}/ca_static.parquet")[["unique_id", "sales_speed"]]
static["unique_id"] = static["unique_id"].astype(str)
first_sale = df[df["y"] > 0].groupby("unique_id")["ds"].min().rename("first_sale").reset_index()

rows = []
for wname, (start, end) in windows.items():
    p_fs, p_zs = f"{OUT_DIR}/{MODEL_NAME}_{wname}.parquet", f"{OUT_DIR}/chronos2_zs_{wname}.parquet"
    if not (os.path.exists(p_fs) and os.path.exists(p_zs)):
        print(f"{wname}: missing {'zero-shot' if os.path.exists(p_fs) else 'few-shot'} file, skipped")
        continue
    fs = pd.read_parquet(p_fs, columns=["unique_id", "ds", "forecast"])
    zs = pd.read_parquet(p_zs, columns=["unique_id", "ds", "forecast"])
    for f_ in (fs, zs):
        f_["unique_id"] = f_["unique_id"].astype(str); f_["ds"] = pd.to_datetime(f_["ds"])
    m = (fs.rename(columns={"forecast": "fs"}).merge(zs.rename(columns={"forecast": "zs"}), on=["unique_id", "ds"])
           .merge(df[["unique_id", "ds", "y"]], on=["unique_id", "ds"])
           .merge(static, on="unique_id", how="left").merge(first_sale, on="unique_id", how="left"))
    m["group"] = m["sales_speed"].astype(str) + " sellers"
    new_mask = (start - m["first_sale"]).dt.days < 365
    parts = [("all products", m)] + [(g, d) for g, d in m.groupby("group")] + [("new products (<1 yr history)", m[new_mask])]
    for name, d in parts:
        if len(d) == 0:
            continue
        a, b = (d["zs"] - d["y"]).abs().mean(), (d["fs"] - d["y"]).abs().mean()
        rows.append({"window": wname, "products": name, "n": d["unique_id"].nunique(),
                     "MAE zero-shot": round(a, 3), "MAE few-shot": round(b, 3), "change": f"{b / a - 1:+.1%}"})
check = pd.DataFrame(rows)
check

,window,products,n,MAE zero-shot,MAE few-shot,change
0,val_1,all products,12196,0.986,0.993,+0.7%
1,val_1,fast sellers,4066,1.959,1.969,+0.6%
2,val_1,medium sellers,4065,0.718,0.727,+1.2%
3,val_1,slow sellers,4065,0.281,0.282,+0.3%
4,val_1,new products (<1 yr history),411,0.860,0.860,-0.1%
5,val_2,all products,12196,0.995,1.004,+0.9%
6,val_2,fast sellers,4066,1.932,1.947,+0.8%
7,val_2,medium sellers,4065,0.751,0.762,+1.4%
8,val_2,slow sellers,4065,0.301,0.302,+0.3%
9,val_2,new products (<1 yr history),379,0.869,0.882,+1.5%


## What to report back
Download this notebook (**File → Download → .ipynb**) with outputs into *Project Code*.
1. **Smoke test:** did it finish? Minutes per window (Section 5).
2. **Full run:** Section 6. Does few-shot beat zero-shot overall? For new products and slow sellers?
3. Then re-run **05** so `chronos2_fs` appears in the official results table (it's included automatically in Friday's run if done by then).